In [1]:
import pandas as pd
from geographiclib.geodesic import Geodesic
import geopandas
import math
import folium
from shapely.geometry import Point
from shapely.geometry import Polygon
import numpy as np
import matplotlib.pyplot as plt
import random as rand
geod = Geodesic.WGS84 

In [2]:
data = 'obc_recreational.csv'
df = pd.read_csv(data)

In [3]:
print(df.dtypes)

MSG_TYPE                 int64
MMSI                     int64
NAME                    object
IMO_NUMBER             float64
CALL_SIGN               object
LAT_AVG                float64
LON_AVG                float64
PERIOD                  object
SPEED_KNOTS            float64
COG_DEG                float64
HEADING_DEG            float64
NAV_STATUS             float64
NAV_SENSOR             float64
SHIP_AND_CARGO_TYPE      int64
DRAUGHT                float64
DRAUGHT.1              float64
DIM_BOW                  int64
DIM_STERN                int64
DIM_PORT                 int64
DIM_STARBOARD            int64
DESTINATION             object
MMSI_COUNTRY_CD         object
RECEIVER                object
dtype: object


In [4]:
df['PERIOD'] = pd.to_datetime(df['PERIOD'])
print(df['PERIOD'].dtypes)

datetime64[ns]


In [5]:
df = df[['MMSI', 'NAME', 'LAT_AVG', 'LON_AVG', 'PERIOD', 'SPEED_KNOTS', 'COG_DEG', 'HEADING_DEG', 'NAV_STATUS', 'SHIP_AND_CARGO_TYPE', 'DRAUGHT', 'DIM_BOW', 'DIM_STERN', 'DIM_PORT', 'DIM_STARBOARD']]
df['BEAM'] = df['DIM_PORT'] + df['DIM_STARBOARD']
df['LENGTH'] = df['DIM_BOW'] + df['DIM_STERN']
df = df[df['BEAM'] > 0]
df = df[df['LENGTH'] > 0]
df = df[df['DRAUGHT'] > 0]
df = df[df['SPEED_KNOTS'] > 3]
df = df.drop_duplicates(subset=['PERIOD', 'MMSI'])

In [6]:
df['channel_side'] = ['Northwestbound' if x > 200 else 'Southeastbound' for x in df['COG_DEG']]
df = df.sort_values(['MMSI', 'PERIOD']).reset_index(drop=True)
df['time_diff'] = (df.groupby('MMSI')['PERIOD'].diff().dt.total_seconds())
#df['cog_diff'] = (df.groupby('MMSI')['COG_DEG'].diff())
#df['cog_diff'] = np.abs((df['cog_diff'] + 180) % 360 -180)

threshold = 30 * 60 # 30 minutes converted to seconds
df['new_voyage'] = (df['time_diff'] > threshold) | (df['time_diff'] < 0 ) | (df['time_diff'].isna()) #| (df['cog_diff'] > 75)
df['voyage_id'] = (df.groupby('MMSI')['new_voyage'].cumsum())
df['voyage_id'] = df['voyage_id'].astype('str') + '_' + df['MMSI'].astype('str')
df

,MMSI,NAME,LAT_AVG,LON_AVG,PERIOD,SPEED_KNOTS,COG_DEG,HEADING_DEG,NAV_STATUS,SHIP_AND_CARGO_TYPE,...,DIM_BOW,DIM_STERN,DIM_PORT,DIM_STARBOARD,BEAM,LENGTH,channel_side,time_diff,new_voyage,voyage_id
0,215224000,AUDACE,22.302193,-77.624380,2025-02-08 11:50:00,12.3,312.4,313.0,0.0,37,...,18,25,5,5,10,43,Northwestbound,NaN,True,1_215224000
1,215224000,AUDACE,22.310921,-77.634689,2025-02-08 11:55:00,12.3,312.4,315.0,0.0,37,...,18,25,5,5,10,43,Northwestbound,300.0,False,1_215224000
2,215224000,AUDACE,22.324641,-77.650895,2025-02-08 12:00:00,12.3,312.7,312.0,0.0,37,...,18,25,5,5,10,43,Northwestbound,300.0,False,1_215224000
3,215224000,AUDACE,22.339261,-77.668200,2025-02-08 12:05:00,12.1,312.1,314.0,0.0,37,...,18,25,5,5,10,43,Northwestbound,300.0,False,1_215224000
4,215224000,AUDACE,22.354541,-77.686284,2025-02-08 12:10:00,12.3,312.5,312.0,0.0,37,...,18,25,5,5,10,43,Northwestbound,300.0,False,1_215224000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5755,538080029,JAMAICA BAY,21.942288,-77.176307,2024-02-19 19:50:00,13.7,312.5,308.0,0.0,37,...,43,17,6,5,11,60,Northwestbound,29413200.0,True,5_538080029
5756,538080029,JAMAICA BAY,21.998985,-77.244164,2024-02-19 20:10:00,12.8,313.0,314.0,0.0,37,...,43,17,6,5,11,60,Northwestbound,1200.0,False,5_538080029
5757,538080029,JAMAICA BAY,22.810750,-78.692764,2024-02-20 03:55:00,10.2,290.6,291.0,0.0,37,...,43,17,6,5,11,60,Northwestbound,27900.0,True,6_538080029
5758,538080029,JAMAICA BAY,22.981145,-79.079337,2024-02-20 06:00:00,10.9,295.0,294.0,0.0,37,...,43,17,6,5,11,60,Northwestbound,7500.0,True,7_538080029


In [7]:
print(f"Total vessels: {df['MMSI'].nunique()}")
print(f"Total voyages: {df['voyage_id'].nunique()}")

Total vessels: 288
Total voyages: 2109


In [8]:
data = pd.read_csv('obc_boundaries.csv')
data['geometry'] = data.apply(lambda x: (float(x.Long), float(x.Lat)), axis=1)
channel_coords = list(data['geometry'])
channel_coords_north = [channel_coords[0], channel_coords[1], channel_coords[2], channel_coords[3], channel_coords[-1], channel_coords[-2], channel_coords[-3], channel_coords[-4]]
channel_coords_south = [channel_coords[4], channel_coords[5], channel_coords[6], channel_coords[7], channel_coords[-1], channel_coords[-2], channel_coords[-3], channel_coords[-4]]
coords_north = [channel_coords[0], channel_coords[1], channel_coords[2], (-77.44235, 22.18378333), (-77.44235, 23.80836667), (-78.72296667, 23.80836667)]
coords_south = [channel_coords[4], channel_coords[5], channel_coords[6], channel_coords[7], (-77.4681,21.15348333), (-78.75143333, 21.15348333)]
channel_polygon_n = Polygon(channel_coords_north)
channel_polygon_s = Polygon(channel_coords_south)
polygon_n = Polygon(coords_north)
polygon_s = Polygon(coords_south)

In [9]:
def where_is_vessel(LAT_AVG, LON_AVG, BOOL_northc, BOOL_southc, BOOL_north, BOOL_south):
    if BOOL_northc == True:
        answer = 'in north channel'
    elif BOOL_southc == True:
        answer = 'in south channel'
    elif LON_AVG < -78.72296667:
        answer = 'west'
    elif LON_AVG > -77.49385:
        answer = 'east'
    elif BOOL_north == True:
        answer = 'north'
    elif BOOL_south == True:
        answer = 'south'
    else:
        answer = 'other'
    return answer

In [10]:
geo_df = geopandas.GeoDataFrame(df, geometry=geopandas.points_from_xy(df['LON_AVG'], df['LAT_AVG']), crs='EPSG:4326')
geo_df['in_channel_north'] = geo_df.within(channel_polygon_n)
geo_df['in_channel_south'] = geo_df.within(channel_polygon_s)
geo_df['north_of_channel'] = geo_df.within(polygon_n)
geo_df['south_of_channel'] = geo_df.within(polygon_s)
geo_df['location'] = geo_df.apply(lambda x: where_is_vessel(x.LAT_AVG, x.LON_AVG, x.in_channel_north, x.in_channel_south, x.north_of_channel, x.south_of_channel), axis=1)
geo_df.shape
geo_df['location'].value_counts()

location
east                1931
in north channel    1442
west                1025
in south channel     798
north                560
south                  4
Name: count, dtype: int64

In [11]:
voyage_list = list(geo_df['voyage_id'].unique())
random_voyages = rand.sample(voyage_list, 40)
geo_df['include'] = geo_df['voyage_id'].isin(random_voyages)
random_voyages_gdf = geo_df[geo_df['include'] == True]
random_voyages_gdf.shape

(99, 28)

In [12]:
data = pd.read_csv('obc_boundaries.csv')
from shapely.geometry import Point
from shapely.geometry import Polygon
data = data[data['Line']!='Middle']
data['geometry'] = data.apply(lambda x: (float(x.Long), float(x.Lat)), axis=1)

channel_coords = list(data['geometry'])
channel_coords_reorder = [channel_coords[0], channel_coords[1], channel_coords[2], channel_coords[3], channel_coords[-1], channel_coords[-2], channel_coords[-3], channel_coords[-4]]
channel_polygon = Polygon(channel_coords_reorder)
geo_df_channel = geopandas.GeoDataFrame(geometry=[channel_polygon], crs='EPSG:4326')

In [13]:
my_map = folium.Map(location=[22.5, -77.8], zoom_start=9.45)
folium.GeoJson(geo_df_channel).add_to(my_map)

def rand_color():
    return "#{:06x}".format(rand.randint(0, 0xFFFFFF))

for id, vessel in random_voyages_gdf.groupby('voyage_id'):
    coords = list(vessel[['LAT_AVG', 'LON_AVG']].itertuples(index=False, name=None))

    folium.PolyLine(
        coords,
        color=rand_color(),
        weight=2,
        opacity=0.8,
        popup=f"MMSI: {id}"
    ).add_to(my_map)

my_map